# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ameen740/Internship_Flyrank/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*


## Research Question

Can observed search-performance signals be used to prioritize content items for human review when there is a potential content refresh opportunity?

## Decision Supported

This analysis supports a content team's decision about which content items should be reviewed first for a possible refresh.

The goal is not to claim that a refresh will improve performance. Instead, the goal is to create a transparent, repeatable way to identify content items that show a potential opportunity based on observed search-performance signals.



In [1]:
research_question = (
    "Can observed search-performance signals be used to prioritize "
    "content items for human review when there is a potential "
    "content refresh opportunity?"
)

decision_supported = (
    "Prioritize content items for human review for a possible refresh."
)

print("Research question:")
print(research_question)

print("\nDecision supported:")
print(decision_supported)

Research question:
Can observed search-performance signals be used to prioritize content items for human review when there is a potential content refresh opportunity?

Decision supported:
Prioritize content items for human review for a possible refresh.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

This analysis uses the FlyRank internship warehouse release available through the public internship dataset.

The unit of analysis is content performance at the report-date, client, and content level.

The analysis uses the March 2026 reporting window. The main search-performance fields used in the analysis are Google Search Console impressions, clicks, and average position.

Client and content identifiers are retained only where needed to identify rows during analysis and are excluded from model features to avoid learning identifier-specific patterns.

Private client names, URLs, search queries, and other identifying information are excluded from the final research output.

Rows without the required search-performance measurements are excluded from the modeling dataset.

In [3]:
# Check what variables are currently available
%whos

Variable             Type    Data/Info
--------------------------------------
decision_supported   str     Prioritize content items <...>w for a possible refresh.
research_question    str     Can observed search-perfo<...>tent refresh opportunity?


In [5]:
# ML-11 — Q2: Data
# Load the March 2026 Search Intelligence data

import duckdb
import pandas as pd
from google.colab import userdata

# Get the Hugging Face token from the Colab Secret
HF_TOKEN = userdata.get("HF_Token")

# Connect to DuckDB
con = duckdb.connect()

# Give DuckDB access to Hugging Face
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret
    (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')
    """
)

# FlyRank internship warehouse
warehouse = "hf://datasets/FlyRank/internship-warehouse"

# Load March 2026 search-performance data
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    gsc_data_available
FROM read_parquet(
    '{warehouse}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
"""

# Create the dataframe
df = con.sql(query).df()

# Basic information
print("Data loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

print("\nDate range:")
print("Start:", df["report_date"].min())
print("End:", df["report_date"].max())

print("\nMissing values:")
print(df.isna().sum())

print("\nExcluded from modeling:")
excluded = [
    "client_hash_id",
    "content_hash_id",
    "future-window information",
    "label-derived information"
]

for item in excluded:
    print("-", item)

# Basic checks
assert len(df) > 0
assert "report_date" in df.columns
assert "gsc_impressions" in df.columns
assert "gsc_clicks" in df.columns

print("\nQ2 checks passed.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data loaded successfully.
Rows: 9841378
Columns: 7

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_data_available']

Date range:
Start: 2026-03-01 00:00:00
End: 2026-03-31 00:00:00

Missing values:
report_date           0
client_hash_id        0
content_hash_id       0
gsc_impressions       0
gsc_clicks            0
gsc_sum_position      0
gsc_data_available    0
dtype: int64

Excluded from modeling:
- client_hash_id
- content_hash_id
- future-window information
- label-derived information

Q2 checks passed.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

# 3. Methodology

The analysis uses observed search-performance signals to identify content that may deserve human review.

## Features

The model uses:

- GSC impressions
- GSC clicks
- CTR
- GSC average position

CTR is calculated as clicks divided by impressions.

## Refresh Opportunity Label

A content item is treated as a refresh-review candidate when it has meaningful search visibility but a low observed CTR.

The baseline rule is:

- impressions >= 100
- CTR < 2%

Rows meeting both conditions are labeled `CTR_FIX_CANDIDATE`. Other rows are labeled `NO_ACTION`.

## Baseline

The transparent rule-based baseline prioritizes content using:

`score = impressions / (CTR + 0.1)`

Higher scores receive higher review priority.

## Model

A Logistic Regression model is used as the modeling approach.

Client identifiers and content identifiers are excluded from model features. Future-window information and label-derived information are also excluded.

## Validation

The model and baseline are evaluated on the same held-out test split.

## Leakage Checks

The model does not use client IDs, content IDs, future performance information, or information derived directly from the target label as predictive features.

In [6]:
import numpy as np

model_df = df.copy()

# Calculate CTR as a percentage
model_df["ctr_pct"] = np.where(
    model_df["gsc_impressions"] > 0,
    (model_df["gsc_clicks"] / model_df["gsc_impressions"]) * 100,
    0
)

# Define refresh opportunity label
model_df["action_label"] = np.where(
    (model_df["gsc_impressions"] >= 100) &
    (model_df["ctr_pct"] < 2.0),
    "CTR_FIX_CANDIDATE",
    "NO_ACTION"
)

# Baseline priority score
model_df["baseline_score"] = (
    model_df["gsc_impressions"] /
    (model_df["ctr_pct"] + 0.1)
)

print(model_df["action_label"].value_counts())


action_label
NO_ACTION            9212433
CTR_FIX_CANDIDATE     628945
Name: count, dtype: int64


In [7]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr_pct",
    "gsc_avg_position"
]

X = model_df[features]
y = model_df["action_label"]

print("Features used:")
print(features)

print("\nFeature shape:", X.shape)
print("Label shape:", y.shape)

KeyError: "['gsc_avg_position'] not in index"

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*


## 5. Limitations

*What this work cannot claim.*


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*



## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
